In [21]:
import pandas as pd
import random            #importing part
import sqlite3
import numpy as np




In [22]:
df=pd.read_csv("data/job_placement_data.csv")   #all reading part
df.head()


,gender,ssc_percentage,ssc_board,hsc_percentage,hsc_board,hsc_subject,degree_percentage,undergrad_degree,work_experience,emp_test_percentage,specialisation,mba_percent,status
0,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed
1,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed
2,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed
3,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed
4,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed


In [23]:
conn=sqlite3.connect("sql.db")
df.to_sql("job_placement",conn,if_exists="replace",index=False)  #connnecting the sql
conn.close()
df.columns=df.columns.str.lower().str.replace(" ","_")
df.info()
                                            

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               215 non-null    object 
 1   ssc_percentage       215 non-null    float64
 2   ssc_board            215 non-null    object 
 3   hsc_percentage       215 non-null    float64
 4   hsc_board            215 non-null    object 
 5   hsc_subject          215 non-null    object 
 6   degree_percentage    215 non-null    float64
 7   undergrad_degree     215 non-null    object 
 8   work_experience      215 non-null    object 
 9   emp_test_percentage  215 non-null    float64
 10  specialisation       215 non-null    object 
 11  mba_percent          215 non-null    float64
 12  status               215 non-null    object 
dtypes: float64(5), object(8)
memory usage: 22.0+ KB


In [24]:
conn=sqlite3.connect("sql.db")
pd.read_sql("SELECT * FROM job_placement LIMIT 5",conn)


,gender,ssc_percentage,ssc_board,hsc_percentage,hsc_board,hsc_subject,degree_percentage,undergrad_degree,work_experience,emp_test_percentage,specialisation,mba_percent,status
0,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed
1,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed
2,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed
3,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed
4,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed


In [25]:
cursor=conn.cursor()
cursor.execute("SELECT COUNT(*) FROM job_placement");      # ans(215,)
cursor.fetchone()  # i made this understand the kpi better 

df_status=pd.read_sql("""
SELECT status, COUNT(*) AS total_students FROM job_placement GROUP BY status
""",conn)
df_totalStudent=pd.read_sql("""
SELECT COUNT(*) AS total_students FROM job_placement""",conn)

df=pd.read_sql("""
SELECT status,COUNT(*) AS count FROM job_placement GROUP BY status                        

""",conn)

df_mba=pd.read_sql("""
SELECT  status, ROUND(AVG(mba_percent),2) AS avg_mba FROM job_placement GROUP BY status""",conn)

df_gender=pd.read_sql("""
SELECT gender, status,COUNT(*) AS count FROM job_placement GROUP BY gender,status""",conn)

df_test=pd.read_sql("""
SELECT ROUND(AVG(emp_test_percentage),2) AS avg_test_score FROM job_placement WHERE status='Placed'""",conn)

print(df_status)


       status  total_students
0  Not Placed              67
1      Placed             148


In [26]:
import pandas as pd
import sqlite3

# connect to DB
conn = sqlite3.connect("sql.db")

# load FULL table (important)
df = pd.read_sql("SELECT * FROM job_placement", conn)

# check columns (sanity check)
print(df.columns.tolist())

# ---- ADD WORK TYPE LOGIC ----
def assign_work_type(row):
    if str(row["work_experience"]).lower() == "yes":
        return "Hybrid"
    elif row["emp_test_percentage"] >= 70:
        return "Remote"
    else:
        return "Offline"

# create new column
df["work_type"] = df.apply(assign_work_type, axis=1)

# verify
print(df["work_type"].value_counts())

# save BACK to sqlite (overwrite table)
df.to_sql("job_placement", conn, if_exists="replace", index=False)



['gender', 'ssc_percentage', 'ssc_board', 'hsc_percentage', 'hsc_board', 'hsc_subject', 'degree_percentage', 'undergrad_degree', 'work_experience', 'emp_test_percentage', 'specialisation', 'mba_percent', 'status']
work_type
Hybrid     74
Remote     72
Offline    69
Name: count, dtype: int64


215

In [27]:
df_check=pd.read_sql(
    "SELECT * FROM job_placement LIMIT 5",conn) # just checked tables
df_check


,gender,ssc_percentage,ssc_board,hsc_percentage,hsc_board,hsc_subject,degree_percentage,undergrad_degree,work_experience,emp_test_percentage,specialisation,mba_percent,status,work_type
0,M,67.00,Others,91.00,Others,Commerce,58.00,Sci&Tech,No,55.0,Mkt&HR,58.80,Placed,Offline
1,M,79.33,Central,78.33,Others,Science,77.48,Sci&Tech,Yes,86.5,Mkt&Fin,66.28,Placed,Hybrid
2,M,65.00,Central,68.00,Central,Arts,64.00,Comm&Mgmt,No,75.0,Mkt&Fin,57.80,Placed,Remote
3,M,56.00,Central,52.00,Central,Science,52.00,Sci&Tech,No,66.0,Mkt&HR,59.43,Not Placed,Offline
4,M,85.80,Central,73.60,Central,Commerce,73.30,Comm&Mgmt,No,96.8,Mkt&Fin,55.50,Placed,Remote


In [28]:

# pd.read_sql("SELECT DISTINCT status FROM job_placement",conn)
pd.read_sql("SELECT COUNT(*) FROM job_placement",conn)
df_hike=pd.read_sql(""" SELECT * FROM job_placement WHERE status ='Placed' AND work_experience ='Yes' """,conn)
df_hike.shape

(64, 14)

In [29]:
def assign_salary_band(row):
    if (
        row["mba_percent"] >= 75 and
        row["emp_test_percentage"] >= 75 and
        row["work_experience"] == "Yes"
    ):
        return "High"
    elif (
        row["mba_percent"] >= 60 or
        row["emp_test_percentage"] >= 60
    ):
        return "Medium"
    else:
        return "Low"

df["salary_band"] = df.apply(assign_salary_band, axis=1)

# Hike Logic 
def assign_hike(row):
    if row["status"] == "Placed" and row["work_experience"] == "Yes":
        if row["salary_band"] == "High":
            return 35
        elif row["salary_band"] == "Medium":
            return 20
        else:
            return 8
    return 0

df["expected_hike_percent"] = df.apply(assign_hike, axis=1)

df.to_sql("job_placement", conn, if_exists="replace", index=False)

conn.close()


df[["salary_band", "expected_hike_percent"]].value_counts()


salary_band  expected_hike_percent
Medium       0                        134
             20                        58
Low          0                         17
             8                          4
High         35                         2
Name: count, dtype: int64

In [30]:
# job switcher data


np.random.seed(42)
n = 200


# (Inspired by industry mobility trends)

experience_years = np.random.choice(
    [1, 2, 3, 4, 5, 7, 10, 15],
    size=n,
    p=[0.15, 0.15, 0.18, 0.17, 0.15, 0.1, 0.07, 0.03]
)


def exp_bucket(x):
    if x <= 2:
        return "0–2 years"
    elif x <= 5:
        return "2–5 years"
    elif x <= 10:
        return "5–10 years"
    else:
        return "10+ years"

experience_bucket = [exp_bucket(x) for x in experience_years]


# Previous 
def role_map(x):
    if x <= 2:
        return "Junior Employee"
    elif x <= 5:
        return "Software Engineer"
    elif x <= 10:
        return "Senior Engineer"
    else:
        return "Team Lead"

previous_role = [role_map(x) for x in experience_years]


primary_skill = np.random.choice(
    ["Python", "SQL", "Java", "Data Analysis", "Web Development"],
    size=n,
    p=[0.3, 0.25, 0.2, 0.15, 0.1]
)


def skill_level(skill):
    if skill in ["Python", "Data Analysis"]:
        return "High"
    elif skill in ["SQL", "Java"]:
        return "Medium"
    else:
        return "Low"

skills_level = [skill_level(s) for s in primary_skill]


# (INSPIRED by your real stats)

def success_rate(exp):
    if exp <= 2:
        return np.random.randint(52, 60)   # ~56%
    elif exp <= 5:
        return np.random.randint(60, 68)   # ~64%
    elif exp <= 10:
        return np.random.randint(45, 52)   # ~49%
    else:
        return np.random.randint(30, 38)   # ~35%

job_switch_success_rate = [success_rate(x) for x in experience_years]


def salary_hike(exp):
    if exp <= 2:
        return np.random.randint(8, 12)
    elif exp <= 5:
        return np.random.randint(12, 18)
    elif exp <= 10:
        return np.random.randint(10, 15)
    else:
        return np.random.randint(8, 12)

expected_salary_increase = [salary_hike(x) for x in experience_years]


job_switch_df = pd.DataFrame({
    "candidate_id": range(1, n + 1),
    "experience_years": experience_years,
    "experience_bucket": experience_bucket,
    "previous_role": previous_role,
    "primary_skill": primary_skill,
    "skills_level": skills_level,
    "job_switch_success_rate": job_switch_success_rate,
    "expected_salary_increase": expected_salary_increase
})


job_switch_df.tail(10)

conn=sqlite3.connect("sql.db")
job_switch_df.to_sql(
    "job_switch_data",
    conn,
    if_exists="replace",
    index=False
)
conn.close()

In [31]:
# =========================
# 2. Create Employee Dataset
# =========================
data = {
    "Employee_ID": [1,2,3,4,5,6,7,8,9,10],
    "Experience_Years": [1,2,3,4,5,6,7,8,9,10],
    "Previous_Salary": [35000,42000,52000,65000,82000,104000,130000,162500,195000,234000],
    "New_Salary": [42000,52000,65000,82000,104000,130000,162500,195000,234000,280800],
    "Salary_Hike_Percent": [20, 23.8, 25, 26.2, 26.8, 25, 25, 20, 20, 20.3],
    "Skill_Level": ["Beginner","Beginner","Intermediate","Intermediate","Advanced",
                    "Advanced","Expert","Expert","Expert","Expert"]
}

df = pd.DataFrame(data)
df

,Employee_ID,Experience_Years,Previous_Salary,New_Salary,Salary_Hike_Percent,Skill_Level
0,1,1,35000,42000,20.0,Beginner
1,2,2,42000,52000,23.8,Beginner
2,3,3,52000,65000,25.0,Intermediate
3,4,4,65000,82000,26.2,Intermediate
4,5,5,82000,104000,26.8,Advanced
5,6,6,104000,130000,25.0,Advanced
6,7,7,130000,162500,25.0,Expert
7,8,8,162500,195000,20.0,Expert
8,9,9,195000,234000,20.0,Expert
9,10,10,234000,280800,20.3,Expert


In [32]:
df["Experience_Bucket"] = pd.cut(
    df["Experience_Years"],
    bins=[0, 2, 5, 9, 100],
    labels=["0–2 years", "2–5 years", "5–10 years", "10+ years"],
    include_lowest=True
)

df

,Employee_ID,Experience_Years,Previous_Salary,New_Salary,Salary_Hike_Percent,Skill_Level,Experience_Bucket
0,1,1,35000,42000,20.0,Beginner,0–2 years
1,2,2,42000,52000,23.8,Beginner,0–2 years
2,3,3,52000,65000,25.0,Intermediate,2–5 years
3,4,4,65000,82000,26.2,Intermediate,2–5 years
4,5,5,82000,104000,26.8,Advanced,2–5 years
5,6,6,104000,130000,25.0,Advanced,5–10 years
6,7,7,130000,162500,25.0,Expert,5–10 years
7,8,8,162500,195000,20.0,Expert,5–10 years
8,9,9,195000,234000,20.0,Expert,5–10 years
9,10,10,234000,280800,20.3,Expert,10+ years


In [33]:
# =========================
# 4. Average Salary Hike by Experience
# =========================
avg_hike = (
    df.groupby("Experience_Bucket",observed=False)["Salary_Hike_Percent"]
    .mean()
    .reset_index()
)

avg_hike

,Experience_Bucket,Salary_Hike_Percent
0,0–2 years,21.9
1,2–5 years,26.0
2,5–10 years,22.5
3,10+ years,20.3


In [38]:
import sqlite3
conn=sqlite3.connect("sql.db")

df.to_sql(
    "salary_hike_experience",
    conn,
    if_exists="replace",
    index=False,
)
conn.close()
import sqlite3

conn = sqlite3.connect("sql.db")
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table';"
).fetchall()
conn.close()

tables


[('company_data',),
 ('job_seeker',),
 ('job placement',),
 ('job_placement',),
 ('job_switch_data',),
 ('salary_hike_experience',)]